# Week 29, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class A:
    def __repr__(self):
        return "A()"

print([A(), A()], str(A()), f"{A()!r}")

**Predict before you run:** what will this print?



In [ ]:
class K:
    def __init__(self, v):
        self.v = v
    def __eq__(self, other):
        return isinstance(other, K) and self.v == other.v

try:
    print(len({K(1), K(1)}))
except TypeError:
    print("TypeError")

**Predict before you run:** what will this print?



In [ ]:
class V:
    def __init__(self, n):
        self.n = n
    def __add__(self, other):
        if not isinstance(other, V):
            return NotImplemented
        return V(self.n + other.n)
    def __radd__(self, other):
        return self if other == 0 else NotImplemented

print(sum([V(1), V(2), V(3)]).n)

**Predict before you run:** what will this print?



In [ ]:
class T:
    def __init__(self, c):
        self.c = c
    @property
    def c(self):
        return self._c
    @c.setter
    def c(self, value):
        self._c = max(value, -273)

print(T(-500).c, vars(T(5)))

**Predict before you run:** what will this print?



In [ ]:
class S:
    def __init__(self):
        self.__x = 1
        self._y = 2

print(sorted(vars(S())))

### Exercise: A score pair

Write `Score(home, away)`, immutable (read-only properties), with `__repr__` like `Score(2, 1)`, `__str__` like `2-1`, equality and hashing, and `+` with another `Score` adding both sides (for aggregate scores over two legs). Also a read-only property `result`: `"home"`, `"away"` or `"draw"`.

In [ ]:
class Score:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    S = ns[\"Score\"]\n    a, b = S(2, 1), S(0, 3)\n    assert repr(a) == \"Score(2, 1)\" and str(a) == \"2-1\" and a + b == S(2, 4) and (a + b).result == \"away\"\n    assert S(1, 1).result == \"draw\" and a.result == \"home\" and len({S(1, 0), S(1, 0)}) == 1\n    for name in (\"home\", \"away\", \"result\"):\n        try:\n            setattr(a, name, 5)\n            assert False, f\"{name} should be read-only\"\n        except AttributeError:\n            pass\n    try:\n        a + 1\n        assert False, \"Score + int should raise TypeError\"\n    except TypeError:\n        pass")

### Exercise: A guest list

Write `GuestList(limit)` with `invite(name)` (names are stripped and title-cased; a duplicate or going over the limit raises `ValueError`), a read-only property `guests` giving a sorted tuple, a property `limit` that can be raised or lowered but never below the number already invited (`ValueError`), and `__repr__` like `<GuestList 3/10>`.

In [ ]:
class GuestList:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"values\"\n    G = ns[\"GuestList\"]\n    g = G(3)\n    g.invite(\" bo \"); g.invite(\"ADA\")\n    assert g.guests == (\"Ada\", \"Bo\") and repr(g) == \"<GuestList 2/3>\"\n    g.limit = 2\n    for f in [lambda: g.invite(\"Cy\"), lambda: g.invite(\"bo\"), lambda: setattr(g, \"limit\", 1)]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    try:\n        g.guests = ()\n        assert False, \"guests should be read-only\"\n    except AttributeError:\n        pass")

### Exercise: Forge: a recipe scaler

Write `Amount`, `Recipe` and `shopping_list` to the specification above.

In [ ]:
class Amount:
    UNITS = {
        "g": ("mass", 1), "kg": ("mass", 1000),
        "ml": ("volume", 1), "l": ("volume", 1000), "tsp": ("volume", 5), "tbsp": ("volume", 15),
        "": ("count", 1),
    }
    ___

class Recipe:
    ___

def shopping_list(recipes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def A(*a):\n    return ns[\"Amount\"](*a)\ndef test_amount_basics():\n    \"amounts\"\n    a = A(1.2, \"kg\")\n    assert (a.value, a.unit, a.family, a.base) == (1.2, \"kg\", \"mass\", 1200)\n    assert repr(A(1.5, \"kg\")) == \"Amount(1.5, 'kg')\" and str(A(1.5, \"kg\")) == \"1.5 kg\" and str(A(3.0, \"tsp\")) == \"3 tsp\"\n    assert str(A(3, \"\")) == \"3\" and str(A(2.333, \"g\")) == \"2.33 g\"\n    for bad in [(1, \"cup\"), (-1, \"g\")]:\n        try:\n            A(*bad)\n            assert False, f\"Amount{bad} should raise ValueError\"\n        except ValueError:\n            pass\n    for name in (\"value\", \"unit\", \"family\", \"base\"):\n        try:\n            setattr(a, name, 1)\n            assert False, f\"{name} should be read-only\"\n        except AttributeError:\n            pass\ndef test_amount_equality():\n    \"equality and hashing\"\n    assert A(1, \"kg\") == A(1000, \"g\") and A(3, \"tsp\") == A(1, \"tbsp\") and A(1, \"l\") != A(1, \"kg\") and A(1, \"g\") != 1\n    assert len({A(1, \"kg\"), A(1000, \"g\"), A(0.1 + 0.2, \"kg\"), A(300, \"g\")}) == 2\ndef test_amount_ops():\n    \"operators\"\n    assert A(500, \"g\") + A(1.2, \"kg\") == A(1.7, \"kg\") and (A(500, \"g\") + A(1.2, \"kg\")).unit == \"g\"\n    assert A(2, \"tbsp\") * 2 == A(60, \"ml\") and 3 * A(2, \"\") == A(6, \"\") and sum([A(1, \"l\"), A(500, \"ml\")]) == A(1.5, \"l\")\n    for f in [lambda: A(1, \"g\") + A(1, \"ml\"), lambda: A(1, \"g\") + 1, lambda: A(1, \"g\") * A(1, \"g\")]:\n        try:\n            f()\n            assert False, \"should raise TypeError\"\n        except TypeError:\n            pass\ndef test_best():\n    \"best format\"\n    assert [f\"{x:best}\" for x in [A(1700, \"g\"), A(250, \"g\"), A(3, \"tsp\"), A(2, \"l\"), A(4, \"\")]] == [\"1.7 kg\", \"250 g\", \"15 ml\", \"2 l\", \"4\"]\ndef test_recipe():\n    \"recipes\"\n    R = ns[\"Recipe\"]\n    r = R(\"Pancakes\", 4)\n    r.add(\"flour\", A(200, \"g\")); r.add(\"milk\", A(300, \"ml\")); r.add(\"eggs\", A(2, \"\")); r.add(\"flour\", A(50, \"g\"))\n    assert r.ingredients == ((\"flour\", A(250, \"g\")), (\"milk\", A(300, \"ml\")), (\"eggs\", A(2, \"\")))\n    assert repr(r) == \"<Recipe 'Pancakes' for 4: 3 ingredients>\"\n    big = r.scaled(6)\n    assert big.servings == 6 and dict(big.ingredients)[\"eggs\"] == A(3, \"\") and dict(r.ingredients)[\"eggs\"] == A(2, \"\")\n    for f, err in [(lambda: r.add(\"milk\", A(1, \"kg\")), TypeError), (lambda: R(\"x\", 0), ValueError), (lambda: setattr(r, \"servings\", 2.5), ValueError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass\n    try:\n        r.ingredients = ()\n        assert False, \"ingredients should be read-only\"\n    except AttributeError:\n        pass\ndef test_shopping():\n    \"shopping list\"\n    R = ns[\"Recipe\"]\n    p = R(\"Pancakes\", 4); p.add(\"flour\", A(200, \"g\")); p.add(\"milk\", A(300, \"ml\")); p.add(\"eggs\", A(2, \"\"))\n    c = R(\"Cake\", 8); c.add(\"flour\", A(1.5, \"kg\")); c.add(\"milk\", A(2, \"tbsp\")); c.add(\"eggs\", A(4, \"\"))\n    assert shopping_list([p, c.scaled(4)]) == [\"eggs: 4\", \"flour: 950 g\", \"milk: 315 ml\"]\n    assert shopping_list([p, c]) == [\"eggs: 6\", \"flour: 1.7 kg\", \"milk: 330 ml\"] and shopping_list([]) == []")

> **Small values, big payoff.** Most of today's work went into `Amount`, a class of about 80 lines that does one thing: be a quantity with a unit. Once it existed, `Recipe` and `shopping_list` hardly needed to think about units at all: they add amounts with `+` and print them with `:best`, and the value class handles every conversion. Good value classes (money, dates, amounts, coordinates) pay for themselves many times over, because every piece of code that uses them gets simpler and safer.

### Exercise: Forge bonus: colours

Write an immutable `Colour(r, g, b)` (each a whole number 0 to 255, or `ValueError`), with:

- read-only properties `r`, `g`, `b`, and `hex` (like `"#ff8000"`) and `brightness` (the perceived brightness `0.299 r + 0.587 g + 0.114 b`, rounded to 1 place);
- class methods `from_hex(text)` (accepting `"#FF8000"`, `"ff8000"`, or the short form `"#f80"`, which means `#ff8800`; `ValueError` otherwise) and `grey(level)`;
- `lighter(fraction)` and `darker(fraction)` returning new colours moved that fraction of the way towards white or black (each channel rounded to the nearest whole number);
- `+` mixing two colours (each channel the rounded average);
- `__repr__` like `Colour(255, 128, 0)`, `__str__` giving the hex form, equality and hashing.

In [ ]:
import re

class Colour:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic():\n    \"basics\"\n    C = ns[\"Colour\"]\n    c = C(255, 128, 0)\n    assert (repr(c), str(c), c.hex, c.brightness) == (\"Colour(255, 128, 0)\", \"#ff8000\", \"#ff8000\", 151.4)\n    assert C.from_hex(\"#FF8000\") == c and C.from_hex(\"ff8000\") == c and C.from_hex(\"#f80\") == C(255, 136, 0)\n    assert C.grey(10) == C(10, 10, 10) and len({c, C(255, 128, 0)}) == 1\ndef test_ops():\n    \"lighter, darker, mixing\"\n    C = ns[\"Colour\"]\n    c = C(255, 128, 0)\n    assert c.lighter(0.5) == C(255, 192, 128) and c.darker(0.5) == C(128, 64, 0) and c == C(255, 128, 0)\n    assert C(0, 0, 0) + C(255, 255, 255) == C(128, 128, 128) and C(10, 20, 30) + C(20, 20, 20) == C(15, 20, 25)\ndef test_errors():\n    \"errors\"\n    C = ns[\"Colour\"]\n    for f in [lambda: C(256, 0, 0), lambda: C(-1, 0, 0), lambda: C(1.5, 0, 0), lambda: C.from_hex(\"#ggg\"), lambda: C.from_hex(\"#ff80\")]:\n        try:\n            f()\n            assert False, \"should raise ValueError\"\n        except ValueError:\n            pass\n    c = C(1, 2, 3)\n    for name in (\"r\", \"hex\", \"brightness\"):\n        try:\n            setattr(c, name, 0)\n            assert False, f\"{name} should be read-only\"\n        except AttributeError:\n            pass")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [repr](w029d1.html), [equality and hashing](w029d2.html), [operators](w029d3.html), [properties](w029d4.html) or [encapsulation](w029d5.html).

## Mixed review

## Forge: a recipe scaler

Recipes are written for a number of servings. Cooking for a different number means scaling every amount, and a shopping list for several recipes means adding up amounts in different units: 500 g of flour and 1.2 kg of flour make 1.7 kg. Here's the specification.

**`Amount(value, unit)`**, an immutable value class.

- Units come in families, in the class attribute `UNITS`: mass (`g` 1, `kg` 1000), volume (`ml` 1, `l` 1000, `tsp` 5, `tbsp` 15), and count (`""`, meaning a number of things, like 3 eggs). Each unit's number is its size in the family's base unit (g or ml). An unknown unit, or a negative value, raises `ValueError`.
- Read-only properties `value`, `unit` and `family`, and `base` (the value in the base unit: `Amount(1.2, "kg").base` is 1200).
- `==` compares **base amounts in the same family**: `Amount(1, "kg") == Amount(1000, "g")`. Different families are never equal. Hashing must agree (hash the family and the base amount, rounded to 6 places to avoid float noise).
- `+` with an `Amount` of the **same family** gives a result in the **left** operand's unit; different families raise `TypeError` (return `NotImplemented`). `sum()` works.
- `*` by a number (either side) scales it, keeping the unit.
- `__repr__` gives `Amount(1.5, 'kg')`. `__str__` gives the value rounded to 2 places with trailing zeros removed, then the unit: `1.5 kg`, `3 tsp`, and just `3` for counts. `__format__` with spec `"best"` gives the str of the amount in the most readable unit of its family: mass of 1000 g or more in kg, otherwise g; volume of 1000 ml or more in l, otherwise ml (teaspoons don't count as most readable); counts unchanged.

**`Recipe(name, servings)`**, with `servings` a positive whole number (a property: `ValueError` otherwise).

- `add(ingredient, amount)`: adding an ingredient already in the recipe **adds** the amounts (raising `TypeError` if their families differ).
- `ingredients`: a read-only property, a **tuple** of `(ingredient, amount)` pairs in the order first added.
- `scaled(servings)`: a **new** recipe for that many servings, every amount scaled.
- `__repr__` gives `<Recipe 'Pancakes' for 4: 3 ingredients>`.

And a function **`shopping_list(recipes)`**: combine the ingredients of all the recipes, and return lines like `flour: 1.7 kg`, sorted by ingredient, each amount shown with the `"best"` format.

## Forge bonus: colours

## Weekly quiz

## Week 29 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write `__repr__` and `__str__`, and know when Python uses each. ([Day 1](w029d1.html))
- I can write `__eq__` and a matching `__hash__`, and decide when a class should be hashable. ([Day 2](w029d2.html))
- I can overload operators sensibly, with `NotImplemented` and reflected methods. ([Day 3](w029d3.html))
- I can use properties for computed, checked and read-only attributes. ([Day 4](w029d4.html))
- I can keep a class's insides private, stop leaks, and make immutable value classes. ([Day 5](w029d5.html))
- I built a recipe scaler and a colour class that behave like Python's own types. (Today)

Tomorrow is a **rest day**. Next week: **inheritance**, where one class builds on another.